In [0]:
%run ../transform/operaciones_replica

In [0]:
TABLA_ORIGEN = "platform_dev.silver_byma.operaciones_replica"
tabla_destino = "platform_dev.gold_byma.dim_cliente"

dbutils.widgets.text("full_load", "0")

full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("dimension_clientes")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(
            f"La tabla {tabla_destino} no existe. "
            "Se ejecutará el notebook de creación de tablas."
        )

        dbutils.notebook.run("../DDL/creacion_tablas", 0)

        carga_full = True

        logger.info(
            "Tablas y esquemas creados correctamente. "
            "Se realizará una carga full."
        )

    elif carga_full:
        logger.warning(
            f"Carga full forzada manualmente para {tabla_destino}."
        )

    else:
        logger.info(f"La tabla {tabla_destino} existe.")

        tiene_datos = spark.sql(
            f"SELECT 1 FROM {tabla_destino} LIMIT 1"
        ).take(1) != []

        if tiene_datos:
            logger.info(
                f"La tabla {tabla_destino} contiene datos. "
                "Se realizará una carga incremental utilizando la tabla de control."
            )
        else:
            carga_full = True

            logger.warning(
                f"La tabla {tabla_destino} está vacía. "
                "Se realizará una carga full."
            )

except Exception:
    logger.exception(
        f"Error al validar o crear la tabla {tabla_destino}"
    )
    raise

In [0]:
logger.info(f"Iniciando lectura de la fuente: {TABLA_ORIGEN}")

# Leemos la tabla Silver base
df_origen = spark.table(TABLA_ORIGEN)

if not carga_full:
    # 1. Traemos la fila completa de control de cargas
    fila_control = spark.sql(f"""
        SELECT ultima_fecha_procesada
        FROM platform_dev.bronce_byma.control_cargas
        WHERE proceso = '{tabla_destino}'
    """).first()

    # 2. Validamos si la fila existe de verdad en la tabla
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
        logger.info(
            f"Carga incremental para {tabla_destino}. "
            f"Última fecha de partición procesada: {ultima_fecha}"
        )
    else:
        ultima_fecha = "1970-01-01"
        logger.warning(
            f"No se encontró registro de control para {tabla_destino} (Arranque en frío). "
            f"Se procesará desde la fecha base: {ultima_fecha}"
        )

    df_clientes_activos = df_origen.filter(
        F.col("fecha_particion") > F.lit(ultima_fecha)
        ).select("id_cliente").distinct()

    df_base_calculo = df_origen.join(df_clientes_activos, "id_cliente", "inner")

    logger.info("Aplicada estrategia de optimización por delta de clientes activos.")

else:
    logger.info(
        f"Carga full para {tabla_destino}. No se aplican filtros de partición."
    )
    df_base_calculo = df_origen

# Ejecutamos las transformaciones y segmentación de negocio
df_transformado = (
    df_base_calculo
    .groupBy("id_cliente")
    .agg(F.count("id_transaccion").alias("total_operaciones"))
    .filter("id_cliente IS NOT NULL")
    .withColumn(
        "segmento_actividad",
        F.when(F.col("total_operaciones") > 30, "Alta frecuencia")
         .when(F.col("total_operaciones") >= 5, "Inversor")
         .otherwise("Esporadico")
    )
    .select("id_cliente", "segmento_actividad")
)

# Registramos la vista temporal para el impacto final en SQL
df_transformado.createOrReplaceTempView("v_clientes_preparados")
logger.info("Transformaciones completadas y vista temporal creada.")

In [0]:
cantidad_registros = df_transformado.count()

try:
    if carga_full:
        logger.warning(
            f"Carga full: Iniciando sobreescritura de {cantidad_registros} registros "
            f"en {tabla_destino} protegiendo la estructura DDL."
        )

        # Usamos INSERT OVERWRITE con declaración explícita de columnas para resguardar la Identity SK
        spark.sql(f"""
            INSERT OVERWRITE {tabla_destino} (id_cliente, segmento_actividad, _created_at)
            SELECT id_cliente, segmento_actividad, current_timestamp()
            FROM v_clientes_preparados
        """)

        logger.info(
            f"Carga full finalizada: {cantidad_registros} registros insertados."
        )

    else:
        if cantidad_registros == 0:
            logger.info("Carga incremental: No se detectaron clientes con actividad nueva.")

        else:
            logger.info(
                f"Carga incremental (MERGE): Procesando {cantidad_registros} clientes modificados/nuevos en {tabla_destino}"
            )

            # Upsert dimensional para mantener estables las claves subrogadas existentes
            spark.sql(f"""
                MERGE INTO {tabla_destino} AS target
                USING v_clientes_preparados AS source
                ON target.id_cliente = source.id_cliente

                WHEN MATCHED AND target.segmento_actividad <> source.segmento_actividad THEN
                  UPDATE SET 
                    target.segmento_actividad = source.segmento_actividad,
                    target._created_at = current_timestamp()

                WHEN NOT MATCHED THEN
                  INSERT (id_cliente, segmento_actividad, _created_at)
                  VALUES (source.id_cliente, source.segmento_actividad, current_timestamp())
            """)

            logger.info(
                f"Carga incremental finalizada con éxito para {cantidad_registros} registros."
            )

except Exception:
    logger.exception(f"Error durante la escritura en {tabla_destino}")
    raise

In [0]:
# Buscamos la fecha máxima de movimiento de la tabla origen para mover la frontera temporal
ultima_fecha_origen = df_origen.agg(F.max("fecha_particion").alias("max_fecha")).first()["max_fecha"]

if ultima_fecha_origen is not None and cantidad_registros > 0:
    try:
        logger.info(
            f"Actualizando watermark del proceso {tabla_destino} "
            f"con fecha de corte: {ultima_fecha_origen}"
        )

        spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (
            SELECT
                '{tabla_destino}' AS proceso,
                DATE('{ultima_fecha_origen}') AS ultima_fecha_procesada,
                current_timestamp() AS fecha_actualizacion
        ) AS source
        ON target.proceso = source.proceso

        WHEN MATCHED THEN UPDATE SET
            target.ultima_fecha_procesada = source.ultima_fecha_procesada,
            target.fecha_actualizacion = source.fecha_actualizacion

        WHEN NOT MATCHED THEN INSERT (
            proceso,
            ultima_fecha_procesada,
            fecha_actualizacion
        )
        VALUES (
            source.proceso,
            source.ultima_fecha_procesada,
            source.fecha_actualizacion
        )
        """)

        logger.info(
            f"Watermark actualizado correctamente para {tabla_destino}."
        )

    except Exception:
        logger.exception(
            f"Error al actualizar la tabla de control para {tabla_destino}"
        )
        raise
else:
    logger.warning(
        "No se modificó el watermark de control debido a que no hubo procesamiento incremental."
    )